# Natural Language Processing

## Exercise Sheet 8

In [1]:
#imports for all exercises
import nltk

from nltk.corpus import treebank
from collections import defaultdict

nltk.download('treebank')

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

### Exercise 1

Write a recursive function to traverse a tree and return the depth of the tree, such that a tree with a single node would have depth zero. (Hint: the depth of a subtree is the maximum depth of its children, plus one.)
Test your function with the two trees produced by the `ChartParser` for the `groucho_grammar` and the sentence "I shot an elephant in my pajamas". The result can be verified with the `Tree.height()` function.



In [2]:
groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")
sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']
parser = nltk.ChartParser(groucho_grammar)
for tree in parser.parse(sent):
    print(tree)

(S
  (NP I)
  (VP
    (VP (V shot) (NP (Det an) (N elephant)))
    (PP (P in) (NP (Det my) (N pajamas)))))
(S
  (NP I)
  (VP
    (V shot)
    (NP (Det an) (N elephant) (PP (P in) (NP (Det my) (N pajamas))))))


In [3]:
# Define the recursive depth function
def tree_depth(tree):
    # Base case: if it's a leaf (string), depth is 0
    if isinstance(tree, str):
        return 0

    # Recursive case: depth is 1 + max depth of children
    if len(tree) == 0:
        return 0

    max_child_depth = max(tree_depth(child) for child in tree)
    return max_child_depth + 1

print("Parsing results:\n")
for i, tree in enumerate(parser.parse(sent), 1):
    print(f"Parse Tree {i}:")
    print(tree)

    # Calculate depth using tree_depth function
    our_depth = tree_depth(tree)

    # Verify with NLTK's height() method
    nltk_height = tree.height()

    print(f"Our depth calculation: {our_depth}")
    print(f"NLTK height() method: {nltk_height}")
    print(f"Match: {our_depth == nltk_height}")
    print("-" * 60)
    print()

Parsing results:

Parse Tree 1:
(S
  (NP I)
  (VP
    (VP (V shot) (NP (Det an) (N elephant)))
    (PP (P in) (NP (Det my) (N pajamas)))))
Our depth calculation: 5
NLTK height() method: 6
Match: False
------------------------------------------------------------

Parse Tree 2:
(S
  (NP I)
  (VP
    (V shot)
    (NP (Det an) (N elephant) (PP (P in) (NP (Det my) (N pajamas))))))
Our depth calculation: 6
NLTK height() method: 7
Match: False
------------------------------------------------------------



### Exercise 2

Write a recursive function `bracketing(tree)` that produces a nested bracketing for a `tree`, leaving out the leaf nodes, and displaying the non-terminal labels after their subtrees. Consecutive categories should be separated by space. Test your function with the tree:

In [4]:
from nltk.corpus import treebank
t = treebank.parsed_sents('wsj_0001.mrg')[0]
print(t)

(S
  (NP-SBJ
    (NP (NNP Pierre) (NNP Vinken))
    (, ,)
    (ADJP (NP (CD 61) (NNS years)) (JJ old))
    (, ,))
  (VP
    (MD will)
    (VP
      (VB join)
      (NP (DT the) (NN board))
      (PP-CLR (IN as) (NP (DT a) (JJ nonexecutive) (NN director)))
      (NP-TMP (NNP Nov.) (CD 29))))
  (. .))


In [5]:
# bracketing(t)
def bracketing(tree):
    # Base case: if it's a leaf (string), return empty string
    if isinstance(tree, str):
        return ""

    # Recursive case: process all children
    child_bracketings = []
    for child in tree:
        child_result = bracketing(child)
        if child_result:  # Only add non-empty results
            child_bracketings.append(child_result)

    # Join children with spaces and wrap in brackets with label
    if child_bracketings:
        inner = " ".join(child_bracketings)
        return f"[{inner} ]{tree.label()}"
    else:
        # If no children produced output (all were leaves), just return the label
        return f"{tree.label()}"



print("Original tree:")
print(t)
print("\n" + "=" * 70 + "\n")

print("Bracketing representation:")
print(bracketing(t))
print("\n" + "=" * 70 + "\n")

Original tree:
(S
  (NP-SBJ
    (NP (NNP Pierre) (NNP Vinken))
    (, ,)
    (ADJP (NP (CD 61) (NNS years)) (JJ old))
    (, ,))
  (VP
    (MD will)
    (VP
      (VB join)
      (NP (DT the) (NN board))
      (PP-CLR (IN as) (NP (DT a) (JJ nonexecutive) (NN director)))
      (NP-TMP (NNP Nov.) (CD 29))))
  (. .))


Bracketing representation:
[[[NNP NNP ]NP , [[CD NNS ]NP JJ ]ADJP , ]NP-SBJ [MD [VB [DT NN ]NP [IN [DT JJ NN ]NP ]PP-CLR [NNP CD ]NP-TMP ]VP ]VP . ]S




Expected output:

    [[[NNP NNP]NP , [[CD NNS]NP JJ]ADJP ,]NP-SBJ [MD [VB [DT NN]NP
    [IN [DT JJ NN]NP]PP-CLR [NNP CD]NP-TMP]VP]VP .]S

### Exercise 3

Modify the functions `init_wfst()` and `complete_wfst()` so that the contents of each cell in the WFST is a set of non-terminal symbols rather than a single non-terminal. Test your function with the `groucho_grammar` and the sentence "I shot an elephant in my pajamas".

In [6]:
groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")
sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']

def init_wfst(tokens, grammar):
    numtokens = len(tokens)
    wfst = [[None for i in range(numtokens+1)] for j in range(numtokens+1)]
    for i in range(numtokens):
        productions = grammar.productions(rhs=tokens[i])
        wfst[i][i+1] = productions[0].lhs()
    return wfst

def complete_wfst(wfst, tokens, grammar, trace=False):
    index = dict((p.rhs(), p.lhs()) for p in grammar.productions())
    numtokens = len(tokens)
    for span in range(2, numtokens+1):
        for start in range(numtokens+1-span):
            end = start + span
            for mid in range(start+1, end):
                nt1, nt2 = wfst[start][mid], wfst[mid][end]
                if nt1 and nt2 and (nt1,nt2) in index:
                    wfst[start][end] = index[(nt1,nt2)]
                    if trace:
                        print("[%s] %3s [%s] %3s [%s] ==> [%s] %3s [%s]" % \
                        (start, nt1, mid, nt2, end, start, index[(nt1,nt2)], end))
    return wfst

def display(wfst):
    print('\nWFST ' + ' '.join(("%-4d" % i) for i in range(1, len(wfst))))
    for i in range(len(wfst)-1):
        print("%d   " % i, end=" ")
        for j in range(1, len(wfst)):
            print("%-4s" % (wfst[i][j] or '.'), end=" ")
        print()

wfst0 = init_wfst(sent, groucho_grammar)
display(wfst0)
wfst = complete_wfst(wfst0, sent, groucho_grammar, True)
display(wfst)


WFST 1    2    3    4    5    6    7   
0    NP   .    .    .    .    .    .    
1    .    V    .    .    .    .    .    
2    .    .    Det  .    .    .    .    
3    .    .    .    N    .    .    .    
4    .    .    .    .    P    .    .    
5    .    .    .    .    .    Det  .    
6    .    .    .    .    .    .    N    
[2] Det [3]   N [4] ==> [2]  NP [4]
[5] Det [6]   N [7] ==> [5]  NP [7]
[1]   V [2]  NP [4] ==> [1]  VP [4]
[4]   P [5]  NP [7] ==> [4]  PP [7]
[0]  NP [1]  VP [4] ==> [0]   S [4]
[1]  VP [4]  PP [7] ==> [1]  VP [7]
[0]  NP [1]  VP [7] ==> [0]   S [7]

WFST 1    2    3    4    5    6    7   
0    NP   .    .    S    .    .    S    
1    .    V    .    VP   .    .    VP   
2    .    .    Det  NP   .    .    .    
3    .    .    .    N    .    .    .    
4    .    .    .    .    P    .    PP   
5    .    .    .    .    .    Det  NP   
6    .    .    .    .    .    .    N    


Change the line:

    NP -> Det N | Det N PP | 'I'

in `groucho_grammar` to:

    NP -> Det N | NP PP | 'I'

to verify in the trace of `complete_wfst()` that there are now two lines for `cell(1,7)`:

    [1]   V [2]  NP [7] ==> [1]  VP [7]
    [1]  VP [4]  PP [7] ==> [1]  VP [7]

In [7]:
#groucho_grammar = ...
#wfst0 = init_wfst(sent, groucho_grammar)
#display(wfst0)
#wfst = complete_wfst(wfst0, sent, groucho_grammar, True)
#display(wfst)

Change the line:

    VP -> V NP | VP PP

in `groucho_grammar` to:

    VP -> V NP
    VPC -> VP PP

and check that `cell(1,7)` now contains `{VPC, VP}`.

In [8]:
#groucho_grammar = ...
#wfst0 = init_wfst(sent, groucho_grammar)
#display(wfst0)
#wfst = complete_wfst(wfst0, sent, groucho_grammar, True)
#display(wfst)

Finally, change the line:

    S -> NP VP

in `groucho_grammar` to:

    S -> NP VP | NP VPC

and check that now there are two lines in the trace of `complete_wfst()` for the `cell(0,7)`:

    [0]  NP [1] VPC [7] ==> [0]   S [7]
    [0]  NP [1]  VP [7] ==> [0]   S [7]

In [9]:
#groucho_grammar = ...
#wfst0 = init_wfst(sent, groucho_grammar)
#display(wfst0)
#wfst = complete_wfst(wfst0, sent, groucho_grammar, True)
#display(wfst)

In [10]:
def init_wfst(tokens, grammar):
    numtokens = len(tokens)
    wfst = [[None for i in range(numtokens+1)] for j in range(numtokens+1)]

    for i in range(numtokens):
        productions = grammar.productions(rhs=tokens[i])
        # Store a SET of all non-terminals that can produce this token
        wfst[i][i+1] = {prod.lhs() for prod in productions}

    return wfst

def complete_wfst(wfst, tokens, grammar, trace=False):
    # Build index: maps (nt1, nt2) -> set of parent non-terminals
    index = {}
    for prod in grammar.productions():
        if len(prod.rhs()) == 2:
            key = tuple(prod.rhs())
            if key not in index:
                index[key] = set()
            index[key].add(prod.lhs())

    numtokens = len(tokens)

    for span in range(2, numtokens+1):
        for start in range(numtokens+1-span):
            end = start + span
            for mid in range(start+1, end):
                nt1_set = wfst[start][mid]
                nt2_set = wfst[mid][end]

                if nt1_set and nt2_set:
                    # Try all combinations of non-terminals from the two cells
                    for nt1 in nt1_set:
                        for nt2 in nt2_set:
                            if (nt1, nt2) in index:
                                # Get all parent non-terminals for this combination
                                parents = index[(nt1, nt2)]

                                # Initialize cell as empty set if needed
                                if wfst[start][end] is None:
                                    wfst[start][end] = set()

                                # Add all parents to the cell
                                for parent in parents:
                                    if parent not in wfst[start][end]:
                                        wfst[start][end].add(parent)
                                        if trace:
                                            print("[%s] %3s [%s] %3s [%s] ==> [%s] %3s [%s]" %
                                                  (start, nt1, mid, nt2, end, start, parent, end))

    return wfst

def display(wfst):
    print('\nWFST ' + ' '.join(("%-8s" % i) for i in range(1, len(wfst))))
    for i in range(len(wfst)-1):
        print("%d   " % i, end=" ")
        for j in range(1, len(wfst)):
            cell = wfst[i][j]
            if cell:
                # Display set as comma-separated string
                cell_str = '{' + ','.join(str(nt) for nt in sorted(cell, key=str)) + '}'
                print("%-8s" % cell_str, end=" ")
            else:
                print("%-8s" % '.', end=" ")
        print()

# Test 1: Original grammar
print("=" * 70)
print("TEST 1: Original grammar")
print("=" * 70)

groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | Det N PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']

wfst0 = init_wfst(sent, groucho_grammar)
display(wfst0)
wfst = complete_wfst(wfst0, sent, groucho_grammar, True)
display(wfst)

# Test 2: Modified NP rule to show two derivations for cell(1,7)
print("\n" + "=" * 70)
print("TEST 2: Modified NP -> Det N | NP PP | 'I'")
print("=" * 70)

groucho_grammar2 = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP | VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

wfst0 = init_wfst(sent, groucho_grammar2)
display(wfst0)
wfst = complete_wfst(wfst0, sent, groucho_grammar2, True)
display(wfst)

# Test 3: Split VP rule to create VPC
print("\n" + "=" * 70)
print("TEST 3: VP -> V NP and VPC -> VP PP")
print("=" * 70)

groucho_grammar3 = nltk.CFG.fromstring("""
S -> NP VP
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP
VPC -> VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

wfst0 = init_wfst(sent, groucho_grammar3)
display(wfst0)
wfst = complete_wfst(wfst0, sent, groucho_grammar3, True)
display(wfst)

# Test 4: Add S -> NP VPC to show two derivations for cell(0,7)
print("\n" + "=" * 70)
print("TEST 4: S -> NP VP | NP VPC")
print("=" * 70)

groucho_grammar4 = nltk.CFG.fromstring("""
S -> NP VP | NP VPC
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP
VPC -> VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

wfst0 = init_wfst(sent, groucho_grammar4)
display(wfst0)
wfst = complete_wfst(wfst0, sent, groucho_grammar4, True)
display(wfst)

TEST 1: Original grammar

WFST 1        2        3        4        5        6        7       
0    {NP}     .        .        .        .        .        .        
1    .        {V}      .        .        .        .        .        
2    .        .        {Det}    .        .        .        .        
3    .        .        .        {N}      .        .        .        
4    .        .        .        .        {P}      .        .        
5    .        .        .        .        .        {Det}    .        
6    .        .        .        .        .        .        {N}      
[2] Det [3]   N [4] ==> [2]  NP [4]
[5] Det [6]   N [7] ==> [5]  NP [7]
[1]   V [2]  NP [4] ==> [1]  VP [4]
[4]   P [5]  NP [7] ==> [4]  PP [7]
[0]  NP [1]  VP [4] ==> [0]   S [4]
[1]  VP [4]  PP [7] ==> [1]  VP [7]
[0]  NP [1]  VP [7] ==> [0]   S [7]

WFST 1        2        3        4        5        6        7       
0    {NP}     .        .        {S}      .        .        {S}      
1    .        {V}      .        {

### Exercise 4

Modify the function `complete_wfst()` from Exercise 3 so that when a non-terminal symbol is added to a cell in the WFST, the content of the variable `mid` is also added, i.e. we add a tuple `(symbol, mid)`. In `init_wfst()`, use `(symbol, i+1)` instead. Change also the function `display()` accordingly. Test your implementation with the final grammar from Exercise 3 and the sentence "I shot an elephant in my pajamas".

In [11]:
def init_wfst(tokens, grammar):
    numtokens = len(tokens)
    wfst = [[None for i in range(numtokens+1)] for j in range(numtokens+1)]

    for i in range(numtokens):
        productions = grammar.productions(rhs=tokens[i])
        # Store a SET of (non-terminal, end_position) tuples
        wfst[i][i+1] = {(prod.lhs(), i+1) for prod in productions}

    return wfst

def complete_wfst(wfst, tokens, grammar, trace=False):
    # Build index: maps (nt1, nt2) -> set of parent non-terminals
    index = {}
    for prod in grammar.productions():
        if len(prod.rhs()) == 2:
            key = tuple(prod.rhs())
            if key not in index:
                index[key] = set()
            index[key].add(prod.lhs())

    numtokens = len(tokens)

    for span in range(2, numtokens+1):
        for start in range(numtokens+1-span):
            end = start + span
            for mid in range(start+1, end):
                nt1_set = wfst[start][mid]
                nt2_set = wfst[mid][end]

                if nt1_set and nt2_set:
                    # Try all combinations of non-terminals from the two cells
                    for nt1_tuple in nt1_set:
                        for nt2_tuple in nt2_set:
                            # Extract just the non-terminal symbols (ignore the mid values)
                            nt1 = nt1_tuple[0]
                            nt2 = nt2_tuple[0]

                            if (nt1, nt2) in index:
                                # Get all parent non-terminals for this combination
                                parents = index[(nt1, nt2)]

                                # Initialize cell as empty set if needed
                                if wfst[start][end] is None:
                                    wfst[start][end] = set()

                                # Add all parents as (parent, mid) tuples to the cell
                                for parent in parents:
                                    parent_tuple = (parent, mid)
                                    if parent_tuple not in wfst[start][end]:
                                        wfst[start][end].add(parent_tuple)
                                        if trace:
                                            print("[%s] %3s [%s] %3s [%s] ==> [%s] %3s [%s]" %
                                                  (start, nt1, mid, nt2, end, start, parent, end))

    return wfst

def display(wfst):
    print('\nWFST', end='')
    for i in range(1, len(wfst)):
        print("%12s" % i, end='')
    print()

    for i in range(len(wfst)-1):
        print("%-4d" % i, end=" ")
        for j in range(1, len(wfst)):
            cell = wfst[i][j]
            if cell:
                # Format as {(NT, mid), (NT, mid), ...}
                tuples_str = ', '.join(f"({nt}, {mid})" for nt, mid in sorted(cell, key=lambda x: (str(x[0]), x[1])))
                cell_str = '{' + tuples_str + '}'
                print("%-12s" % cell_str, end=" ")
            else:
                print("%-12s" % '.', end=" ")
        print()

# Test with the final grammar from Exercise 3
print("=" * 100)
print("WFST with (symbol, mid) tuples - Final grammar: S -> NP VP | NP VPC")
print("=" * 100)

groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP | NP VPC
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP
VPC -> VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']

wfst0 = init_wfst(sent, groucho_grammar)
print("\nAfter initialization:")
display(wfst0)

print("\n\nTrace of complete_wfst():")
wfst = complete_wfst(wfst0, sent, groucho_grammar, True)

print("\n\nFinal WFST:")
display(wfst)

WFST with (symbol, mid) tuples - Final grammar: S -> NP VP | NP VPC

After initialization:

WFST           1           2           3           4           5           6           7
0    {(NP, 1)}    .            .            .            .            .            .            
1    .            {(V, 2)}     .            .            .            .            .            
2    .            .            {(Det, 3)}   .            .            .            .            
3    .            .            .            {(N, 4)}     .            .            .            
4    .            .            .            .            {(P, 5)}     .            .            
5    .            .            .            .            .            {(Det, 6)}   .            
6    .            .            .            .            .            .            {(N, 7)}     


Trace of complete_wfst():
[2] Det [3]   N [4] ==> [2]  NP [4]
[5] Det [6]   N [7] ==> [5]  NP [7]
[1]   V [2]  NP [4] ==> [1]  VP [4]
[4] 

It should produce the following output:

    WFST      1          2          3          4          5          6          7         
    0         {(NP, 1)}  .          .          {(S, 1)}   .          .          {(S, 1)}   
    1         .          {(V, 2)}   .          {(VP, 2)}  .          .          {(VP, 2), (VPC, 4)}
    2         .          .          {(Det, 3)} {(NP, 3)}  .          .          {(NP, 4)}  
    3         .          .          .          {(N, 4)}   .          .          .          
    4         .          .          .          .          {(P, 5)}   .          {(PP, 5)}  
    5         .          .          .          .          .          {(Det, 6)} {(NP, 6)}  
    6         .          .          .          .          .          .          {(N, 7)}    

### Exercise 5

Use the extended WFST from Exercise 4 to retrace the parse trees for our example sentence "I shot an elephant in my pajamas''. Write a recursive function `retrace(WFST, tokens)` (the second parameter `tokens` contains the token list \['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas'\] for our example sentence). Start with `cell(0,7)` (or `cell(0,len(tokens))` in general) and use the information in `mid` to follow the productions to `cell(0,mid)` and `cell(mid,7)`, and so on. If we reach a terminal symbol, i.e. a `cell(i,i+1)`, the corresponding token from `tokens` shall be displayed.

In [12]:
def init_wfst(tokens, grammar):
    numtokens = len(tokens)
    wfst = [[None for i in range(numtokens+1)] for j in range(numtokens+1)]

    for i in range(numtokens):
        productions = grammar.productions(rhs=tokens[i])
        wfst[i][i+1] = {(prod.lhs(), i+1) for prod in productions}

    return wfst

def complete_wfst(wfst, tokens, grammar, trace=False):
    index = {}
    for prod in grammar.productions():
        if len(prod.rhs()) == 2:
            key = tuple(prod.rhs())
            if key not in index:
                index[key] = set()
            index[key].add(prod.lhs())

    numtokens = len(tokens)

    for span in range(2, numtokens+1):
        for start in range(numtokens+1-span):
            end = start + span
            for mid in range(start+1, end):
                nt1_set = wfst[start][mid]
                nt2_set = wfst[mid][end]

                if nt1_set and nt2_set:
                    for nt1_tuple in nt1_set:
                        for nt2_tuple in nt2_set:
                            nt1 = nt1_tuple[0]
                            nt2 = nt2_tuple[0]

                            if (nt1, nt2) in index:
                                parents = index[(nt1, nt2)]

                                if wfst[start][end] is None:
                                    wfst[start][end] = set()

                                for parent in parents:
                                    parent_tuple = (parent, mid)
                                    if parent_tuple not in wfst[start][end]:
                                        wfst[start][end].add(parent_tuple)
                                        if trace:
                                            print("[%s] %3s [%s] %3s [%s] ==> [%s] %3s [%s]" %
                                                  (start, nt1, mid, nt2, end, start, parent, end))

    return wfst

def retrace(wfst, tokens, grammar):
    start = 0
    end = len(tokens)

    cell = wfst[start][end]

    if cell is None:
        print("No parse found.")
        return

    # Display each possible parse tree
    for symbol, mid in sorted(cell, key=lambda x: (str(x[0]), x[1])):
        _display_tree(wfst, tokens, grammar, start, end, symbol, mid, 0)
        print()  # Blank line between different parse trees

def _display_tree(wfst, tokens, grammar, start, end, symbol, mid, indent):
    # Print the current symbol with proper indentation
    print(" " * indent + str(symbol), end="")

    # Base case: preterminal (span of 1 token)
    if end - start == 1:
        # This is a leaf node, print the terminal
        print("   -> " + tokens[start])
        return

    # Recursive case: non-terminal with children
    print("   ->", end="")

    # Get the cells for left and right children
    left_cell = wfst[start][mid]
    right_cell = wfst[mid][end]

    # Find the production rule that created this symbol
    for prod in grammar.productions(lhs=symbol):
        if len(prod.rhs()) == 2:
            left_nt, right_nt = prod.rhs()

            # Find matching symbols in the left and right cells
            left_matches = [(s, m) for s, m in left_cell if s == left_nt]
            right_matches = [(s, m) for s, m in right_cell if s == right_nt]

            if left_matches and right_matches:
                # Use the first match found
                left_symbol, left_mid = left_matches[0]
                right_symbol, right_mid = right_matches[0]

                # Print newline and recursively display children
                print()
                _display_tree(wfst, tokens, grammar, start, mid, left_symbol, left_mid, indent + 5)
                _display_tree(wfst, tokens, grammar, mid, end, right_symbol, right_mid, indent + 5)
                return

groucho_grammar = nltk.CFG.fromstring("""
S -> NP VP | NP VPC
PP -> P NP
NP -> Det N | NP PP | 'I'
VP -> V NP
VPC -> VP PP
Det -> 'an' | 'my'
N -> 'elephant' | 'pajamas'
V -> 'shot'
P -> 'in'
""")

sent = ['I', 'shot', 'an', 'elephant', 'in', 'my', 'pajamas']

# Build the WFST
wfst0 = init_wfst(sent, groucho_grammar)
wfst = complete_wfst(wfst0, sent, groucho_grammar, False)

# Retrace and display all parse trees
print("\nParse trees:\n")
retrace(wfst, sent, groucho_grammar)


Parse trees:

S   ->
     NP   -> I
     VP   ->
          V   -> shot
          NP   ->
               NP   ->
                    Det   -> an
                    N   -> elephant
               PP   ->
                    P   -> in
                    NP   ->
                         Det   -> my
                         N   -> pajamas



### Exercise 6

Process each tree of the Penn Treebank Corpus sample `nltk.corpus.treebank` and extract the productions with the help of `Tree.productions()`. Discard the productions that occur only once and those that are lexical (i.e. the right-hand side contains at least one terminal token). Productions with the same left-hand side can be collapsed using a dictionary with the left-hand sides as keys and sets of right-hand sides as values.

Print the value for the left-hand side 'NP' using the format:

    DT JJS NN NN | DT VBG NN NN | DT NNP CD NN | DT NN NNS ...


In [13]:
def extract_grammar_from_treebank():
    # Count all productions
    production_counts = defaultdict(int)

    # Process each tree in the treebank
    for tree in treebank.parsed_sents():
        productions = tree.productions()
        for prod in productions:
            production_counts[prod] += 1

    # Filter productions:
    # 1. Must occur more than once
    # 2. Must be non-lexical (no terminals on RHS)
    filtered_productions = []
    for prod, count in production_counts.items():
        if count > 1:
            # Check if production is non-lexical
            # A production is lexical if RHS contains any terminal (string)
            is_lexical = any(isinstance(symbol, str) for symbol in prod.rhs())
            if not is_lexical:
                filtered_productions.append(prod)

    # Collapse productions by LHS
    # Dictionary with LHS as keys and sets of RHS as values
    grammar_dict = defaultdict(set)
    for prod in filtered_productions:
        lhs = prod.lhs()
        rhs = prod.rhs()  # This is a tuple of symbols
        grammar_dict[lhs].add(rhs)

    return grammar_dict

# Extract the grammar
print("Extracting productions from Penn Treebank...")
print("=" * 80)

grammar = extract_grammar_from_treebank()

# Print statistics
print(f"\nTotal non-terminals with non-lexical productions: {len(grammar)}")
print(f"Total unique non-lexical productions: {sum(len(rhs_set) for rhs_set in grammar.values())}")

# Print NP productions in the requested format
print("\n" + "=" * 80)
print("Productions for 'NP':")
print("=" * 80)

np_symbol = nltk.Nonterminal('NP')
if np_symbol in grammar:
    np_productions = grammar[np_symbol]

    # Convert each RHS tuple to a string with spaces between symbols
    rhs_strings = []
    for rhs in sorted(np_productions):
        rhs_str = ' '.join(str(symbol) for symbol in rhs)
        rhs_strings.append(rhs_str)

    # Join with " | " separator
    output = ' | '.join(rhs_strings)
    print(output)

    print(f"\n\nTotal number of NP production rules: {len(np_productions)}")
else:
    print("NP not found in grammar")

Extracting productions from Penn Treebank...

Total non-terminals with non-lexical productions: 184
Total unique non-lexical productions: 2667

Productions for 'NP':
$ CD -NONE- | -NONE- | : NP PP , SBAR . | ADJP DT NN | ADJP JJ NN | ADJP JJ NNS | ADJP NN | ADJP NN NN | ADJP NN NNS | ADJP NNP | ADJP NNP NNP | ADJP NNP NNP NNP | ADJP NNP NNS | ADJP NNS | CD | CD CC CD | CD CD | CD CD NN | CD JJ JJ NNS | CD JJ NN | CD JJ NN NNS | CD JJ NNS | CD JJR NNS | CD NN | CD NN JJ NNS | CD NN NN | CD NN NN NNS | CD NN NNS | CD NN QP | CD NNP NNS | CD NNS | CD NP NNS | CD VBN NNS | DT | DT $ CD -NONE- | DT ADJP JJ NN | DT ADJP JJ NN NN | DT ADJP NN | DT ADJP NN NN | DT ADJP NN NNS | DT ADJP NNP | DT ADJP NNP NN | DT ADJP NNP NNP NN | DT ADJP NNS | DT ADJP NNS NN | DT ADJP QP -NONE- | DT CD | DT CD CC CD | DT CD CD | DT CD JJ JJ NNS | DT CD JJ NNS | DT CD NN | DT CD NN NN | DT CD NN NNS | DT CD NNS | DT CD VBN NNS | DT JJ | DT JJ , JJ NN | DT JJ , JJ NNS | DT JJ CC JJ NNS | DT JJ CD | DT JJ CD NN | 